In [ ]:
import requests

url = (
    "https://ru.wikipedia.org"
    "/api/rest_v1/page/html/"
    "Уравнение_Ван-дер-Ваальса"
)

response = requests.get(url)


In [ ]:
%pip install lxml

In [1]:
import pandas as pd

In [ ]:
df = pd.read_html(
    response.text, 
    decimal=",", 
    thousands=None,
)[0]

df.to_csv("Константы_Ван-дер-Ваальса.tsv", sep="\t", index=False)

In [9]:
pd.read_csv(
    "./Константы_Ван-дер-Ваальса.tsv", 
    sep="\t", 
    index_col="Вещество"
).head(3)

,Unnamed: 0,"a, Па·м6·моль−2","b, 10−6 м3·моль−1"
Вещество,,,
Азот N2,0,0.1370,38.7
Аммиак NH3,1,0.4225,37.1
Аргон Ar,2,0.1355,32.0


In [12]:
df = pd.read_csv(
    "./Константы_Ван-дер-Ваальса.tsv", 
    sep="\t", 
    index_col=0
)
df.head()

,Вещество,"a, Па·м6·моль−2","b, 10−6 м3·моль−1"
0,Азот N2,0.1370,38.7
1,Аммиак NH3,0.4225,37.1
2,Аргон Ar,0.1355,32.0
3,Ацетилен C2H2,0.4516,52.2
4,Бром Br2,0.9750,59.1


In [22]:
df_expand = df.copy()
df_expand[["Вещество", "Formula"]] = df["Вещество"]\
    .str.rsplit(" ", n=1, expand=True)
df_expand.head(2)

,Вещество,"a, Па·м6·моль−2","b, 10−6 м3·моль−1",Formula
0,Азот,0.1370,38.7,N2
1,Аммиак,0.4225,37.1,NH3


In [27]:
df.columns

Index(['Вещество', 'a,  Па·м6·моль−2', 'b,  10−6 м3·моль−1'], dtype='object')

In [36]:
df_expand = df.copy()
df_expand[["Name", "Formula"]] = df["Вещество"]\
    .str.rsplit(" ", n=1, expand=True)
df_expand.drop(columns="Вещество", inplace=True)
df_expand.rename(
    columns={"a,  Па·м6·моль−2": "A", 'b,  10−6 м3·моль−1': "B",},
    inplace=True
)
df_expand.head(1)

,A,B,Name,Formula
0,0.137,38.7,Азот,N2


In [35]:
df_expand.to_csv("./Константы_Ван-дер-Ваальса_красивые.tsv",sep="\t")

## Pubchem

In [38]:
%pip install pubchempy ssl pyodide-http

In [39]:
import pyodide_http
pyodide_http.patch_all()

In [40]:
import pubchempy

In [42]:
h2_compounds = pubchempy.get_compounds(
    "H2", namespace="formula"
)
h2_compounds

[Compound(783),
 Compound(24523),
 Compound(24824),
 Compound(167583),
 Compound(5460631),
 Compound(119434),
 Compound(6914304),
 Compound(6914290),
 Compound(157679700),
 Compound(159167674),
 Compound(169430093)]

In [51]:
h2_compounds[0].bonds

[Bond(1, 2, 1)]

In [48]:
h2_compounds[0].molecular_weight

2.016

In [52]:
import pandas as pd

In [55]:
df_pretty = pd.read_csv(
    "./Константы_Ван-дер-Ваальса_красивые.tsv",
    sep="\t",
    index_col=0
).head(2)

df_pretty

,A,B,Name,Formula
0,0.1370,38.7,Азот,N2
1,0.4225,37.1,Аммиак,NH3


In [56]:

# df_pretty["Formula"]

0     N2
1    NH3
Name: Formula, dtype: object

In [57]:
df_pretty_with_mw = df_pretty.copy()
df_pretty_with_mw["Molecular_weight"] = df_pretty.Formula.map(
    lambda f:  pubchempy.get_compounds(f, namespace="formula")[0]\
        .molecular_weight
)
df_pretty_with_mw

,A,B,Name,Formula,Molecular_weight
0,0.1370,38.7,Азот,N2,28.014
1,0.4225,37.1,Аммиак,NH3,17.031
